In [0]:

# ============================================================
# P3 CANDLESTICK REVERSAL v3.0 — CONTROL RUN (EURUSD H4 2020-2024)
# ============================================================
# PROTOCOL DEVIATIONS LOG (Research Override — see cell output):
#   1. Method 3 Stop Order Validity: 2 candles → 4 candles
#   2. Dual-Regime RSI Gate: RSI<35 (reversal) + RSI<50 (pullback)
#   3. Isolated signal: No ADX, Volume, Liquidity, or Macro filters applied
#
# PURPOSE: Establish raw pattern edge baseline before filter optimization.
# ARCHITECTURE: Preserved from Protocol Engine v2.5.2 (P3StrictValidator).

import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURATION & ASSET DNA
# ==========================================
MT5_PATH = r"C:\Program Files\SCFM MT5 Terminal\terminal64.exe"

# Symbol mapping: internal key → MT5 symbol
SYMBOL_MAP = {
    'EURUSD=X': 'EURUSD.pro', 'GBPUSD=X': 'GBPUSD.pro', 'USDJPY=X': 'USDJPY.pro',
    'AUDUSD=X': 'AUDUSD.pro', 'NZDUSD=X': 'NZDUSD.pro', 'USDCAD=X': 'USDCAD.pro',
    'USDCHF=X': 'USDCHF.pro', 'EURGBP=X': 'EURGBP.pro',
    'GC=F': 'XAUUSD.pro',  'SI=F': 'XAGUSD.pro',  'USOIL': 'USOIL',
    'NAS100': 'NAS100.pro',  'SPX500': 'SP500.pro',   'US30': 'WS30.pro'
}

ASSET_DNA = {
    'EURUSD=X': {
        'pip': 0.0001, 'fee_pip': 2.0, 'atr_mult': 1.0, 'sl_buf_pip': 1.5,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'GBPUSD=X': {
        'pip': 0.0001, 'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'USDJPY=X': {
        'pip': 0.01,   'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'AUDUSD=X': {
        'pip': 0.0001, 'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'NZDUSD=X': {
        'pip': 0.0001, 'fee_pip': 3.0, 'atr_mult': 1.0, 'sl_buf_pip': 1.5,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'USDCAD=X': {
        'pip': 0.0001, 'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'USDCHF=X': {
        'pip': 0.0001, 'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'EURGBP=X': {
        'pip': 0.0001, 'fee_pip': 3.0, 'atr_mult': 1.25, 'sl_buf_pip': 1.5,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'GC=F': {
        'pip': 0.01,   'fee_pip': 3.5, 'atr_mult': 1.5, 'sl_buf_pip': 5.0,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'SI=F': {
        'pip': 0.001,  'fee_pip': 4.5, 'atr_mult': 1.75, 'sl_buf_pip': 0.03,
        'rsi_pullback': 50, 'rsi_reversal': 20, 'rsi_bear': 80
    },
    'USOIL': {
        'pip': 0.01,   'fee_pip': 3.0, 'atr_mult': 1.0, 'sl_buf_pip': 0.02,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'NAS100': {
        'pip': 0.1,    'fee_pip': 2.5, 'atr_mult': 1.5, 'sl_buf_pip': 0.50,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'SPX500': {
        'pip': 0.1,    'fee_pip': 2.5, 'atr_mult': 1.5, 'sl_buf_pip': 0.50,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    },
    'US30': {
        'pip': 0.1,    'fee_pip': 2.5, 'atr_mult': 1.5, 'sl_buf_pip': 0.50,
        'rsi_pullback': 50, 'rsi_reversal': 35, 'rsi_bear': 65
    }
}

# Protocol deviation flag — logged in output
DEVIATION_LOG = {
    'STOP_ORDER_VALIDITY': '2 → 4 candles (Method 3 extension for research)',
    'RSI_GATE': 'Dual-regime: RSI<35 (reversal) + RSI<50 (pullback) — research override',
    'FILTERS_DISABLED': 'ADX, Volume, Liquidity, Macro — isolated core signal only',
    'LOOKAHEAD_AUDIT': 'Confirmation uses C+1.close after pattern at C — NOT before entry. CLEAN.',
    'TIMEFRAME': 'H4, 2020-01-01 to 2024-12-31 (EURUSD control)'
}

def fetch_mt5_h4(symbol, start_date, end_date, path=MT5_PATH):
    """Fetch H4 data for a specific date range."""
    if not mt5.initialize(path=path):
        raise Runtime("MT5 Init Failed")
    if not mt5.symbol_select(symbol, True):
        return pd.DataFrame()
    
    # Convert dates to UTC timestamps
    utc_from = datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=__import__('pytz').utc) if 'pytz' in globals() else pd.Timestamp(start_date, tz='UTC')
    utc_to = datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=__import__('pytz').utc) if 'pytz' in globals() else pd.Timestamp(end_date, tz='UTC')
    
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_H4, utc_from, utc_to)
    if rates is None or len(rates) == 0:
        return pd.DataFrame()
    df = pd.DataFrame(rates)
    df["time"] = pd.to_datetime(df["time"], unit="s")
    df = df.set_index("time")[["open", "high", "low", "close", "tick_volume"]]
    df = df.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "tick_volume": "Volume"})
    return df


# ============================================================
# 2. INDICATOR ENGINE (Wilder's Smoothing)
# ============================================================
def wilders_atr(df, period=14):
    """Average True Range with Wilder's smoothing (RMA)."""
    high, low, close = df['High'], df['Low'], df['Close']
    tr1 = high - low
    tr2 = abs(high - close.shift(1))
    tr3 = abs(low - close.shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    # Wilder's RMA = EWM with alpha = 1/period
    atr = tr.ewm(alpha=1/period, adjust=False).mean()
    return atr

def wilders_rsi(df, period=14):
    """RSI with Wilder's smoothing (TradingView/MT5 compatible)."""
    delta = df['Close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    # Wilder's smoothing: alpha = 1/period
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    with np.errstate(divide='ignore', invalid='ignore'):
        rs = avg_gain / avg_loss
        rsi = 100 - (100 / (1 + rs))
    rsi = rsi.replace([np.inf, -np.inf], 100).fillna(50)
    return rsi


# ============================================================
# 3. PATTERN DETECTION (P3 v3.0 Core Signal)
# ============================================================
def detect_patterns(df):
    """
    Detect hammer and engulfing patterns.
    Returns boolean Series for each pattern type.
    """
    o, c, h, l = df['Open'], df['Close'], df['High'], df['Low']
    body = (c - o).abs().replace(0, np.nan)
    u_wick = h - np.maximum(o, c)
    l_wick = np.minimum(o, c) - l
    crange = h - l

    # Hammer: lower wick >= 2× body, upper wick <= 0.25× body, close in top 25%
    is_hammer = (l_wick >= 2 * body) & (u_wick <= 0.25 * body) & (c >= l + 0.75 * crange)
    is_hammer = is_hammer.fillna(False)

    # Bullish Engulfing
    prev_body = body.shift(1)
    is_bull_engulf = (c > o) & (c.shift(1) < o.shift(1)) & (c > o.shift(1)) & (o < c.shift(1)) & (body > prev_body)
    is_bull_engulf = is_bull_engulf.fillna(False)

    # Shooting Star (bearish hammer)
    is_star = (u_wick >= 2 * body) & (l_wick <= 0.25 * body) & (c <= h - 0.75 * crange)
    is_star = is_star.fillna(False)

    # Bearish Engulfing
    is_bear_engulf = (c < o) & (c.shift(1) > o.shift(1)) & (c < o.shift(1)) & (o > c.shift(1)) & (body > prev_body)
    is_bear_engulf = is_bear_engulf.fillna(False)

    return {
        'hammer': is_hammer,
        'bull_engulf': is_bull_engulf,
        'star': is_star,
        'bear_engulf': is_bear_engulf,
        'body': body,
        'u_wick': u_wick,
        'l_wick': l_wick,
        'range': crange
    }


# ============================================================
# 4. BACKTEST ENGINE (Event-Driven Simulation)
# ============================================================
class P3ControlRun:
    """
    P3 Candlestick Reversal Control Run — Isolated Signal Baseline.

    Protocol Deviations (research overrides):
        - Stop order validity: 4 candles (was 2)
        - Dual-regime RSI: <35 (reversal) + <50 (pullback)
        - No ADX, volume, liquidity, or macro filters
    """

    def __init__(self, symbol, start='2020-01-01', end='2024-12-31',
                 stop_validity=4, use_dual_regime=True,
                 isolated_signal=True):  # if True, skip ADX/volume/macro
        self.symbol = symbol
        self.start = start
        self.end = end
        self.stop_validity = stop_validity
        self.use_dual_regime = use_dual_regime
        self.isolated_signal = isolated_signal
        self.dna = ASSET_DNA.get(symbol, ASSET_DNA['EURUSD=X'])
        self.df = None
        self.trades = []

    def fetch_and_prepare(self):
        """Fetch data, compute indicators, prepare for backtest."""
        sym_mt5 = SYMBOL_MAP.get(self.symbol)
        if not sym_mt5:
            raise ValueError(f"No MT5 mapping for {self.symbol}")

        print(f"▶ Fetching {self.symbol} H4 [{self.start} → {self.end}]...")
        df = fetch_mt5_h4(sym_mt5, self.start, self.end)
        if df.empty:
            raise ValueError(f"No data fetched for {self.symbol}")

        # Timezone to London for session filtering
        df.index = df.index.tz_localize('UTC').tz_convert('Europe/London')

        # Indicators (Wilder's smoothing)
        df['ATR'] = wilders_atr(df, period=14)
        df['RSI'] = wilders_rsi(df, period=14)
        df['EMA200'] = df['Close'].ewm(span=200, adjust=False).mean()

        # Session filter (London/NY overlap, weekdays, no Friday afternoon)
        is_weekday = df.index.weekday < 5
        is_fri_pm = (df.index.weekday == 4) & (df.index.hour >= 16)
        df['Session_OK'] = is_weekday & ~is_fri_pm & (df.index.hour >= 8) & (df.index.hour < 21)

        self.df = df
        print(f"  ✓ {len(df)} candles | {df.index[0]} → {df.index[-1]}")

    def run_backtest(self):
        """Execute the event-driven backtest."""
        if self.df is None:
            self.fetch_and_prepare()

        df = self.df
        dna = self.dna

        # Pattern detection
        patterns = detect_patterns(df)
        is_hammer = patterns['hammer']
        is_bull_engulf = patterns['bull_engulf']
        is_star = patterns['star']
        is_bear_engulf = patterns['bear_engulf']

        # Trend + Volatility (kept for context, not filter if isolated)
        trend_bull = df['Close'] > df['EMA200']
        trend_bear = df['Close'] < df['EMA200']
        vol_gate = (df['High'] - df['Low']) > (df['ATR'] * dna['atr_mult'])

        # --- DUAL-REGIME RSI GATE (Research Override) ---
        if self.use_dual_regime:
            # Pullback: price > EMA200, RSI < 50 (relaxed from < 35)
            bull_rsi_pullback = trend_bull & (df['RSI'] < dna['rsi_pullback'])
            # Reversal: price < EMA200, RSI < 35 (deep oversold)
            bull_rsi_reversal = trend_bear & (df['RSI'] < dna['rsi_reversal'])
            bull_rsi_gate = bull_rsi_pullback | bull_rsi_reversal

            # Bearish mirrors
            bear_rsi_pullback = trend_bear & (df['RSI'] > (100 - dna['rsi_pullback']))
            bear_rsi_reversal = trend_bull & (df['RSI'] > (100 - dna['rsi_reversal']))
            bear_rsi_gate = bear_rsi_pullback | bear_rsi_reversal
        else:
            # Legacy single-regime (strict)
            bull_rsi_gate = trend_bull & (df['RSI'] < dna['rsi_reversal'])
            bear_rsi_gate = trend_bear & (df['RSI'] > dna['rsi_bear'])

        # Core signal (isolated: pattern + RSI + session only)
        if self.isolated_signal:
            valid_bull = ((is_hammer | is_bull_engulf) & bull_rsi_gate & df['Session_OK']).fillna(False)
            valid_bear = ((is_star | is_bear_engulf) & bear_rsi_gate & df['Session_OK']).fillna(False)
        else:
            # Full filter stack (for comparison)
            # Note: ADX calculation omitted for isolated run
            valid_bull = ((is_hammer | is_bull_engulf) & bull_rsi_gate & vol_gate & df['Session_OK']).fillna(False)
            valid_bear = ((is_star | is_bear_engulf) & bear_rsi_gate & vol_gate & df['Session_OK']).fillna(False)

        signals = (valid_bull | valid_bear).fillna(False)
        print(f"\n  Raw signals: {signals.sum()} | Bull: {valid_bull.sum()} | Bear: {valid_bear.sum()}" )

        # --- EVENT-DRIVEN TRADE SIMULATION ---
        self.trades = []
        idx_list = list(df.index)

        for idx in signals[signals].index:
            i = idx_list.index(idx)
            if i + 15 >= len(df):  # Need room for confirmation + trade management
                continue

            direction = 'Bull' if valid_bull.iloc[i] else 'Bear'
            p_high, p_low = df['High'].iloc[i], df['Low'].iloc[i]

            # --- STEP 1: CONFIRMATION (C+1 closes beyond pattern extreme) ---
            c1_close = df['Close'].iloc[i + 1]
            c1_high = df['High'].iloc[i + 1]
            c1_low = df['Low'].iloc[i + 1]

            confirmed = False
            if direction == 'Bull' and c1_close > p_high:
                confirmed = True
            if direction == 'Bear' and c1_close < p_low:
                confirmed = True
            if not confirmed:
                continue

            # --- STEP 2: EXHAUSTION FILTER (C+1 range < 2.0× ATR) ---
            if (c1_high - c1_low) > (2.0 * df['ATR'].iloc[i]):
                continue

            # --- STEP 3: STOP ORDER TRIGGER (Method 3, extended validity) ---
            buf = 1.5 * dna['pip']
            trigger = (c1_high + buf) if direction == 'Bull' else (c1_low - buf)
            triggered = False
            entry_idx = None

            # DEVIATION: stop validity extended from 2 to 4 candles
            for t in range(2, 2 + self.stop_validity):
                if i + t >= len(df):
                    break
                if direction == 'Bull' and df['High'].iloc[i + t] > trigger:
                    triggered = True
                    entry_idx = i + t
                    break
                if direction == 'Bear' and df['Low'].iloc[i + t] < trigger:
                    triggered = True
                    entry_idx = i + t
                    break
            if not triggered:
                continue

            # --- STEP 4: STATEFUL TRADE SIMULATION (P3 v3.0 Exit Rules) ---
            entry_price = trigger
            sl_buf = dna['sl_buf_pip'] * dna['pip']

            if direction == 'Bull':
                risk = entry_price - (p_low - sl_buf)
                tp1 = entry_price + 2 * risk  # 2R
                tp2 = entry_price + 3 * risk  # 3R
            else:
                risk = (p_high + sl_buf) - entry_price
                tp1 = entry_price - 2 * risk
                tp2 = entry_price - 3 * risk

            if risk <= 0:
                continue

            # Fee approximation: 0.05R spread + 0.05R commission
            fee_r = 0.10
            final_r = -1.0  # Default loss
            trade_active = True
            max_unrealized_r = 0.0
            partial_closed = False
            realized_r = 0.0
            current_sl = entry_price - risk if direction == 'Bull' else entry_price + risk

            start_j = entry_idx + 1
            end_j = min(start_j + 10, len(df))  # Time stop: 10 candles from entry

            for j in range(start_j, end_j):
                hv = df['High'].iloc[j]
                lv = df['Low'].iloc[j]
                cv = df['Close'].iloc[j]

                if direction == 'Bull':
                    current_r = (hv - entry_price) / risk
                    if lv <= current_sl:
                        remaining_r = (current_sl - entry_price) / risk
                        final_r = realized_r + 0.5 * remaining_r
                        trade_active = False
                        break
                    if hv >= tp1:
                        current_r = max(current_r, 2.0)
                    if hv >= tp2:
                        current_r = max(current_r, 3.0)
                else:
                    current_r = (entry_price - lv) / risk
                    if hv >= current_sl:
                        remaining_r = (entry_price - current_sl) / risk
                        final_r = realized_r + 0.5 * remaining_r
                        trade_active = False
                        break
                    if lv <= tp1:
                        current_r = max(current_r, 2.0)
                    if lv <= tp2:
                        current_r = max(current_r, 3.0)

                if current_r > max_unrealized_r:
                    max_unrealized_r = current_r

                # P3 RULE: BE at 1R
                if max_unrealized_r >= 1.0:
                    current_sl = entry_price

                # P3 RULE: Partial Close at 2R, Trail SL to +1R
                if not partial_closed and max_unrealized_r >= 2.0:
                    partial_closed = True
                    realized_r = 1.0
                    current_sl = entry_price + risk if direction == 'Bull' else entry_price - risk

                # P3 RULE: TP2 / Runner Exit
                if max_unrealized_r >= 3.0:
                    realized_r += 1.5
                    final_r = realized_r
                    trade_active = False
                    break

            # Time stop execution
            if trade_active and j == end_j - 1:
                exit_p = df['Close'].iloc[j]
                r_market = (exit_p - entry_price) / risk if direction == 'Bull' else (entry_price - exit_p) / risk
                final_r = realized_r + 0.5 * max(r_market, -1.0)

            # Deduct fees
            final_r -= fee_r
            self.trades.append({
                'entry_time': df.index[entry_idx],
                'direction': direction,
                'entry_price': entry_price,
                'risk': risk,
                'final_r': final_r,
                'regime': 'Pullback' if (direction == 'Bull' and valid_bull.iloc[i] and trend_bull.iloc[i]) or (direction == 'Bear' and valid_bear.iloc[i] and trend_bear.iloc[i]) else 'Reversal',
                'entry_type': 'StopOrder'
            })

        return self.trades

    def report(self):
        """Generate statistical report."""
        trades = self.trades
        if not trades:
            print("\n❌ NO TRADES EXECUTED." )
            return None

        r_vals = np.array([t['final_r'] for t in trades])
        wins = r_vals[r_vals > 0]
        losses = r_vals[r_vals <= 0]
        n = len(trades)
        win_rate = len(wins) / n if n > 0 else 0.0
        avg_r = np.mean(r_vals) if n > 0 else 0.0
        profit_factor = abs(np.sum(wins) / np.sum(losses)) if len(losses) > 0 and np.sum(losses) != 0 else float('inf')

        # Wilson 95% CI Lower Bound
        if n > 0:
            z = 1.96
            p = win_rate
            wilson_lb = (p + z**2/(2*n) - z*np.sqrt((p*(1-p) + z**2/(4*n))/n)) / (1 + z**2/n)
            wilson_lb = max(0, wilson_lb)  # Clamp to [0, 1]
        else:
            wilson_lb = 0.0

        # Regime/entry breakdown
        pullback_trades = [t for t in trades if t['regime'] == 'Pullback']
        reversal_trades = [t for t in trades if t['regime'] == 'Reversal']

        print("\n" + "="*80)
        print(f"📊 P3 CONTROL RUN REPORT — {self.symbol} H4 [{self.start} → {self.end}]")
        print("="*80)
        print(f"  N (Total Trades):     {n}")
        print(f"  Win Rate:             {win_rate:.1%}")
        print(f"  Avg R-Multiple:       {avg_r:.3f}")
        print(f"  Profit Factor:        {profit_factor:.2f}")
        print(f"  Wilson 95% CI LB:     {wilson_lb:.1%}")
        print(f"\n  --- Regime Breakdown ---")
        print(f"  Pullback (RSI<50):    {len(pullback_trades)} trades")
        print(f"  Reversal (RSI<35):    {len(reversal_trades)} trades")
        print(f"\n  --- Entry Type ---")
        print(f"  Stop Order (Method 3): {len(trades)} trades")
        print("="*80)

        # Decision gate
        print("\n🚦 DECISION GATE:")
        if n >= 15 and wilson_lb >= 0.48:
            print("   ✅ PASS → Proceed to Signal-First Baseline (Step 2)")
        elif n >= 15:
            print(f"   ⚠️  MARGINAL — N≥15 but Wilson CI_LB < 48% ({wilson_lb:.1%}). Tighten execution or relax filters.")
        elif n >= 10:
            print(f"   ⚠️  LOW SAMPLE — N={n}. Consider longer period or lower timeframe.")
        else:
            print(f"   ❌ FAIL — N={n} < 10. Recommend H4 retirement or Daily/H1 shift.")

        # Protocol deviations log
        print("\n📋 PROTOCOL DEVIATIONS (Research Overrides):")
        for k, v in DEVIATION_LOG.items():
            print(f"   {k}: {v}")

        return {
            'n': n,
            'win_rate': win_rate,
            'avg_r': avg_r,
            'profit_factor': profit_factor,
            'wilson_lb': wilson_lb,
            'pullback_n': len(pullback_trades),
            'reversal_n': len(reversal_trades),
            'trades': trades
        }


# ============================================================
# 5. EXECUTION: EURUSD CONTROL RUN
# ============================================================
if __name__ == "__main__":
    # Initialize MT5
    if not mt5.initialize(path=MT5_PATH):
        raise RuntimeError("MT5 Init Failed")

    try:
        runner = P3ControlRun(
            symbol='EURUSD=X',
            start='2020-01-01',
            end='2024-12-31',
            stop_validity=4,       # DEVIATION: extended from 2
            use_dual_regime=True,  # DEVIATION: RSI<35 + RSI<50
            isolated_signal=True   # No ADX/volume/macro filters
        )
        runner.run_backtest()
        result = runner.report()
    finally:
        mt5.shutdown()
"""
